# CBAM-ResNet50 Training - IMPROVED v2

**Model:** ResNet-50 with Convolutional Block Attention Module (CBAM)  
**Attention:** Dual attention mechanism (channel + spatial)  
**Dataset:** Kermany OCT2017 (patient-stratified, verified clean)  
**Validation:** 15% stratified split (11,521 images)

## CBAM Architecture

CBAM applies sequential channel and spatial attention:
1. **Channel Attention:** Recalibrates feature maps by channel importance
2. **Spatial Attention:** Emphasizes informative spatial regions

Applied after each ResNet stage for hierarchical attention learning.

In [1]:
# IMPORTS
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
from torchvision import transforms, models
from torchvision.datasets import ImageFolder
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, precision_score, recall_score
from pathlib import Path
import numpy as np
import time
from tqdm import tqdm
from collections import Counter
import json
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

print("Imports successful")

Imports successful


In [2]:
# HELPER FUNCTIONS

def get_next_serial_number(checkpoint_dir):
    """Automatically detect the next available serial number for checkpoints."""
    import re
    from pathlib import Path
    
    checkpoint_dir = Path(checkpoint_dir)
    if not checkpoint_dir.exists():
        checkpoint_dir.mkdir(parents=True, exist_ok=True)
        return 1
    
    existing = list(checkpoint_dir.glob("*.pth"))
    if not existing:
        return 1
    
    serial_numbers = []
    for f in existing:
        match = re.match(r'^(\d+)_', f.name)
        if match:
            serial_numbers.append(int(match.group(1)))
    
    return max(serial_numbers) + 1 if serial_numbers else 1


def save_checkpoint(model, optimizer, epoch, metrics, is_best, checkpoint_dir,
                   serial_number, model_name, seed, mode='intermediate'):
    """Save model checkpoint with comprehensive training state and metrics."""
    from datetime import datetime
    from pathlib import Path
    
    checkpoint_dir = Path(checkpoint_dir)
    checkpoint_dir.mkdir(parents=True, exist_ok=True)
    
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    serial_str = f"{serial_number:02d}"
    
    filename = f"{serial_str}_{model_name}_seed{seed}_epoch{epoch}_{mode}_{timestamp}.pth"
    filepath = checkpoint_dir / filename
    
    checkpoint = {
        'serial_number': serial_number,
        'model_name': model_name,
        'seed': seed,
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'metrics': metrics,
        'is_best': is_best,
        'mode': mode,
        'timestamp': timestamp
    }
    
    torch.save(checkpoint, filepath)
    print(f"Saved {mode}: {filename}")
    return filepath


def create_stratified_split(dataset, val_ratio=0.15, seed=42):
    """
    Create stratified train/validation split maintaining class balance.
    Uses pre-loaded labels from ImageFolder.targets for efficiency.
    """
    labels = np.array(dataset.targets)
    indices = np.arange(len(labels))
    
    train_idx, val_idx = train_test_split(
        indices,
        test_size=val_ratio,
        stratify=labels,
        random_state=seed
    )
    
    return train_idx, val_idx


def is_better_model(new_score, new_loss, new_acc, new_epoch,
                    best_score, best_loss, best_acc, best_epoch,
                    eps=1e-9):
    """
    Deterministic model comparison with clear priority hierarchy.
    
    Priority order:
    1. Composite score (primary metric)
    2. Validation loss (tie-breaker)
    3. Validation accuracy (secondary tie-breaker)
    4. Epoch number (prefer later epochs for stability)
    
    Returns True if new model outperforms current best.
    """
    if new_score > best_score + eps:
        return True
    
    if abs(new_score - best_score) <= eps:
        if new_loss < best_loss - eps:
            return True
        
        if abs(new_loss - best_loss) <= eps:
            if new_acc > best_acc + eps:
                return True
            
            if abs(new_acc - best_acc) <= eps:
                if new_epoch > best_epoch:
                    return True
    
    return False


def check_overfitting(train_acc, val_acc, train_loss, val_loss, 
                     threshold_acc=10.0, threshold_loss=0.5):
    """Detect overfitting based on train-validation performance gaps."""
    acc_gap = train_acc - val_acc
    loss_gap = val_loss - train_loss
    
    is_overfitting = (acc_gap > threshold_acc) or (loss_gap > threshold_loss)
    
    return {
        'is_overfitting': is_overfitting,
        'acc_gap': acc_gap,
        'loss_gap': loss_gap,
        'severity': 'HIGH' if (acc_gap > 15.0 or loss_gap > 1.0) else 'MODERATE' if is_overfitting else 'NONE'
    }


print("Helper functions loaded")

Helper functions loaded


In [3]:
# CONFIGURATION

ROOT = Path(r"C:\Users\Ajant\Documents\Project MSc - CSC-40098\3 - Experiments\Notebooks\Model_Training")

CHECKPOINT_DIR = ROOT / "Checkpoints"
DATASET_ROOT = ROOT / "Data_Kermany_OCT2017"
TRAIN_PATH = DATASET_ROOT / "train"
TEST_PATH = DATASET_ROOT / "test"

MODEL_NAME = "cbam_resnet"
NUM_EPOCHS = 50
SEED = 126

BATCH_SIZE = 32
LEARNING_RATE = 0.001
WEIGHT_DECAY = 1e-4
IMAGE_SIZE = 224
NUM_CLASSES = 4
CLASS_NAMES = ['CNV', 'DME', 'DRUSEN', 'NORMAL']

VAL_SPLIT_RATIO = 0.15
SAVE_EVERY_N_EPOCHS = 5
OVERFITTING_CHECK_INTERVAL = 5

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Set seeds for reproducibility
torch.manual_seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

SERIAL_NUMBER = get_next_serial_number(CHECKPOINT_DIR)

print("="*80)
print("CONFIGURATION - CBAM-RESNET50")
print("="*80)
print(f"Model: {MODEL_NAME}")
print(f"Serial: {SERIAL_NUMBER:02d} | Seed: {SEED} | Epochs: {NUM_EPOCHS}")
print(f"Device: {DEVICE}")
print(f"Validation: {VAL_SPLIT_RATIO*100:.0f}% stratified split")
print("="*80)

CONFIGURATION - CBAM-RESNET50
Model: cbam_resnet
Serial: 09 | Seed: 126 | Epochs: 50
Device: cuda
Validation: 15% stratified split


In [4]:
# DATASET VERIFICATION

print("="*80)
print("VERIFYING DATASET INTEGRITY")
print("="*80)

def list_files(root):
    """Get set of all image filenames in directory."""
    return set([p.name for p in Path(root).rglob("*.jpeg")])

train_files = list_files(TRAIN_PATH)
test_files = list_files(TEST_PATH)

print(f"Train files: {len(train_files):,}")
print(f"Test files: {len(test_files):,}")

overlap = train_files.intersection(test_files)
print(f"Overlap check: {len(overlap)} files")

if len(overlap) > 0:
    print("❌ WARNING: Train/test overlap detected!")
    print("Examples:", list(overlap)[:10])
    raise ValueError("Dataset contains train/test overlap")
else:
    print("✅ No overlap - dataset is clean")

print("="*80)

VERIFYING DATASET INTEGRITY
Train files: 55,792
Test files: 968
Overlap check: 0 files
✅ No overlap - dataset is clean


In [5]:
# DATASET LOADING

print("\n" + "="*80)
print("CREATING STRATIFIED TRAIN/VAL SPLIT")
print("="*80)

# Data transforms
train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=10),
    transforms.ColorJitter(brightness=0.1, contrast=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_test_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Load dataset for stratification
full_dataset = ImageFolder(root=str(TRAIN_PATH))
print(f"Total training images: {len(full_dataset):,}")

# Create stratified split
train_idx, val_idx = create_stratified_split(full_dataset, VAL_SPLIT_RATIO, SEED)

print(f"\nSplit created:")
print(f"  Training: {len(train_idx):,} images ({(1-VAL_SPLIT_RATIO)*100:.1f}%)")
print(f"  Validation: {len(val_idx):,} images ({VAL_SPLIT_RATIO*100:.1f}%)")

# Verify class balance
train_labels = [full_dataset.targets[i] for i in train_idx]
val_labels = [full_dataset.targets[i] for i in val_idx]

train_counts = Counter(train_labels)
val_counts = Counter(val_labels)

print("\nClass distribution:")
print(f"{'Class':<12} {'Training':>10} {'Validation':>12} {'Val %':>8}")
print("-" * 50)
for i, class_name in enumerate(CLASS_NAMES):
    train_count = train_counts[i]
    val_count = val_counts[i]
    val_pct = (val_count / (train_count + val_count)) * 100
    print(f"{class_name:<12} {train_count:>10,} {val_count:>12,} {val_pct:>7.1f}%")

# Create datasets with transforms
train_dataset_full = ImageFolder(root=str(TRAIN_PATH), transform=train_transform)
val_dataset_full = ImageFolder(root=str(TRAIN_PATH), transform=val_test_transform)

train_dataset = Subset(train_dataset_full, train_idx)
val_dataset = Subset(val_dataset_full, val_idx)

# Create dataloaders
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=4,
    pin_memory=True if torch.cuda.is_available() else False
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=4,
    pin_memory=True if torch.cuda.is_available() else False
)

print(f"\nDataLoaders created:")
print(f"  Train batches: {len(train_loader)}")
print(f"  Val batches: {len(val_loader)}")
print("="*80)


CREATING STRATIFIED TRAIN/VAL SPLIT
Total training images: 55,792

Split created:
  Training: 47,423 images (85.0%)
  Validation: 8,369 images (15.0%)

Class distribution:
Class          Training   Validation    Val %
--------------------------------------------------
CNV              19,006        3,354    15.0%
DME               5,862        1,034    15.0%
DRUSEN            3,280          579    15.0%
NORMAL           19,275        3,402    15.0%

DataLoaders created:
  Train batches: 1482
  Val batches: 262


In [6]:
# CBAM MODEL ARCHITECTURE

class ChannelAttention(nn.Module):
    """
    Channel Attention Module.
    
    Recalibrates channel-wise feature responses by explicitly modeling
    interdependencies between channels using global pooling and MLPs.
    """
    def __init__(self, channels, reduction=16):
        super(ChannelAttention, self).__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)
        
        # Shared MLP for both pooling paths
        self.mlp = nn.Sequential(
            nn.Linear(channels, channels // reduction, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(channels // reduction, channels, bias=False)
        )
        self.sigmoid = nn.Sigmoid()
    
    def forward(self, x):
        b, c, _, _ = x.size()
        
        # Average pooling path
        avg_out = self.mlp(self.avg_pool(x).view(b, c))
        
        # Max pooling path
        max_out = self.mlp(self.max_pool(x).view(b, c))
        
        # Combine and apply sigmoid
        out = self.sigmoid(avg_out + max_out).view(b, c, 1, 1)
        
        return x * out.expand_as(x)


class SpatialAttention(nn.Module):
    """
    Spatial Attention Module.
    
    Focuses on informative spatial regions by aggregating channel information
    through pooling and applying spatial convolution.
    """
    def __init__(self, kernel_size=7):
        super(SpatialAttention, self).__init__()
        self.conv = nn.Conv2d(2, 1, kernel_size, padding=kernel_size//2, bias=False)
        self.sigmoid = nn.Sigmoid()
    
    def forward(self, x):
        # Channel-wise pooling
        avg_out = torch.mean(x, dim=1, keepdim=True)
        max_out, _ = torch.max(x, dim=1, keepdim=True)
        
        # Concatenate and convolve
        out = torch.cat([avg_out, max_out], dim=1)
        out = self.sigmoid(self.conv(out))
        
        return x * out


class CBAM(nn.Module):
    """
    Convolutional Block Attention Module.
    
    Sequentially applies channel and spatial attention to refine features
    along both dimensions. Channel attention is applied first, followed by
    spatial attention on the channel-refined features.
    """
    def __init__(self, channels, reduction=16, kernel_size=7):
        super(CBAM, self).__init__()
        self.channel_attention = ChannelAttention(channels, reduction)
        self.spatial_attention = SpatialAttention(kernel_size)
    
    def forward(self, x):
        x = self.channel_attention(x)
        x = self.spatial_attention(x)
        return x


class CBAMResNet50(nn.Module):
    """
    ResNet-50 with CBAM attention modules.
    
    Integrates CBAM after each residual stage to enable hierarchical attention
    learning. CBAM modules refine features at multiple scales, from low-level
    textures to high-level semantic patterns.
    """
    def __init__(self, num_classes=4, pretrained=True, reduction=16):
        super(CBAMResNet50, self).__init__()
        
        # Load pretrained ResNet-50 backbone
        resnet = models.resnet50(pretrained=pretrained)
        
        # Copy backbone layers
        self.conv1 = resnet.conv1
        self.bn1 = resnet.bn1
        self.relu = resnet.relu
        self.maxpool = resnet.maxpool
        
        self.layer1 = resnet.layer1
        self.layer2 = resnet.layer2
        self.layer3 = resnet.layer3
        self.layer4 = resnet.layer4
        
        # Add CBAM modules after each stage
        self.cbam1 = CBAM(256, reduction)   # After layer1 (256 channels)
        self.cbam2 = CBAM(512, reduction)   # After layer2 (512 channels)
        self.cbam3 = CBAM(1024, reduction)  # After layer3 (1024 channels)
        self.cbam4 = CBAM(2048, reduction)  # After layer4 (2048 channels)
        
        self.avgpool = resnet.avgpool
        self.fc = nn.Linear(2048, num_classes)
    
    def forward(self, x):
        # Initial convolution
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.maxpool(x)
        
        # ResNet stages with CBAM
        x = self.layer1(x)
        x = self.cbam1(x)
        
        x = self.layer2(x)
        x = self.cbam2(x)
        
        x = self.layer3(x)
        x = self.cbam3(x)
        
        x = self.layer4(x)
        x = self.cbam4(x)
        
        # Classification head
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        x = self.fc(x)
        
        return x


print("CBAM architecture defined")

CBAM architecture defined


In [7]:
# MODEL INITIALIZATION

model = CBAMResNet50(num_classes=NUM_CLASSES, pretrained=True, reduction=16)
model = model.to(DEVICE)

# Class-balanced loss
class_weights = torch.tensor([
    len(train_labels) / (NUM_CLASSES * train_counts[i])
    for i in range(NUM_CLASSES)
], dtype=torch.float32).to(DEVICE)

criterion = nn.CrossEntropyLoss(weight=class_weights)

# Optimizer
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

# Learning rate scheduler
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=5
)

print("="*80)
print("MODEL INITIALIZED")
print("="*80)
print(f"Architecture: CBAM-ResNet50")
print(f"Parameters: ~{sum(p.numel() for p in model.parameters()) / 1e6:.1f}M")
print(f"CBAM reduction ratio: 16")
print(f"Class weights: {class_weights.cpu().numpy()}")
print("="*80)

MODEL INITIALIZED
Architecture: CBAM-ResNet50
Parameters: ~24.2M
CBAM reduction ratio: 16
Class weights: [0.62378985 2.0224752  3.614558   0.6150843 ]


In [8]:
# TRAINING LOOP

print("\n" + "="*80)
print(f"STARTING TRAINING - {MODEL_NAME.upper()}")
print("="*80)
print(f"Serial: {SERIAL_NUMBER:02d} | Seed: {SEED} | Epochs: {NUM_EPOCHS}")
print(f"Device: {DEVICE}")
print(f"Val size: {len(val_dataset):,} images")
print("="*80)

# Training history
history = {
    'train_loss': [], 'train_acc': [],
    'val_loss': [], 'val_acc': [],
    'val_f1': [], 'val_precision': [], 'val_recall': [],
    'composite_score': [],
    'learning_rates': [],
    'overfitting_checks': []
}

# Initialize best model tracking
best_composite_score = float('-inf')
best_val_acc = 0.0
best_val_loss = float('inf')
best_epoch = -1

start_time = time.time()

try:
    for epoch in range(NUM_EPOCHS):
        epoch_start = time.time()
        
        print(f"\nEpoch [{epoch+1}/{NUM_EPOCHS}]")
        print("-" * 70)
        
        # TRAINING PHASE
        model.train()
        train_loss = 0.0
        train_correct = 0
        train_total = 0
        
        for images, labels in tqdm(train_loader, desc="Training", leave=False):
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item() * images.size(0)
            _, predicted = torch.max(outputs, 1)
            train_total += labels.size(0)
            train_correct += (predicted == labels).sum().item()
        
        train_loss = train_loss / len(train_dataset)
        train_acc = 100.0 * train_correct / train_total
        
        # VALIDATION PHASE
        model.eval()
        val_loss = 0.0
        all_preds = []
        all_labels = []
        
        with torch.no_grad():
            for images, labels in tqdm(val_loader, desc="Validation", leave=False):
                images, labels = images.to(DEVICE), labels.to(DEVICE)
                
                outputs = model(images)
                loss = criterion(outputs, labels)
                
                val_loss += loss.item() * images.size(0)
                _, predicted = torch.max(outputs, 1)
                
                all_preds.extend(predicted.cpu().numpy())
                all_labels.extend(labels.cpu().numpy())
        
        val_loss = val_loss / len(val_dataset)
        val_acc = 100.0 * np.mean(np.array(all_preds) == np.array(all_labels))
        
        # Validate accuracy scale
        assert 0 <= train_acc <= 100, f"Train acc {train_acc:.2f} out of range"
        assert 0 <= val_acc <= 100, f"Val acc {val_acc:.2f} out of range"
        
        # Compute additional metrics
        val_f1 = f1_score(all_labels, all_preds, average='macro') * 100
        val_precision = precision_score(all_labels, all_preds, average='macro', zero_division=0) * 100
        val_recall = recall_score(all_labels, all_preds, average='macro', zero_division=0) * 100
        
        # Composite score for model selection
        composite_score = (
            0.40 * val_acc +
            0.25 * val_f1 +
            0.20 * (100 - min(val_loss * 10, 100)) +
            0.15 * max(0, 100 - abs(train_acc - val_acc) * 2)
        )
        
        # Update history
        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)
        history['val_f1'].append(val_f1)
        history['val_precision'].append(val_precision)
        history['val_recall'].append(val_recall)
        history['composite_score'].append(composite_score)
        history['learning_rates'].append(optimizer.param_groups[0]['lr'])
        
        scheduler.step(val_loss)
        
        # Best model selection
        is_best = is_better_model(
            new_score=composite_score,
            new_loss=val_loss,
            new_acc=val_acc,
            new_epoch=epoch + 1,
            best_score=best_composite_score,
            best_loss=best_val_loss,
            best_acc=best_val_acc,
            best_epoch=best_epoch
        )
        
        if is_best:
            best_composite_score = composite_score
            best_val_acc = val_acc
            best_val_loss = val_loss
            best_epoch = epoch + 1
            
            metrics = {
                'train_loss': train_loss, 'train_acc': train_acc,
                'val_loss': val_loss, 'val_acc': val_acc,
                'val_f1': val_f1, 'val_precision': val_precision, 'val_recall': val_recall,
                'composite_score': composite_score
            }
            
            save_checkpoint(model, optimizer, epoch + 1, metrics, True,
                          CHECKPOINT_DIR, SERIAL_NUMBER, MODEL_NAME, SEED, 'best')
        
        # Periodic checkpoints
        if (epoch + 1) % SAVE_EVERY_N_EPOCHS == 0:
            metrics = {
                'train_loss': train_loss, 'train_acc': train_acc,
                'val_loss': val_loss, 'val_acc': val_acc,
                'val_f1': val_f1, 'val_precision': val_precision, 'val_recall': val_recall,
                'composite_score': composite_score
            }
            
            save_checkpoint(model, optimizer, epoch + 1, metrics, False,
                          CHECKPOINT_DIR, SERIAL_NUMBER, MODEL_NAME, SEED, 'intermediate')
        
        # Overfitting monitoring
        if (epoch + 1) % OVERFITTING_CHECK_INTERVAL == 0:
            overfit_check = check_overfitting(train_acc, val_acc, train_loss, val_loss)
            history['overfitting_checks'].append((epoch + 1, overfit_check))
            
            if overfit_check['is_overfitting']:
                print(f"\n⚠️  OVERFITTING WARNING [{overfit_check['severity']}]:")
                print(f"   Train-Val Acc Gap: {overfit_check['acc_gap']:.2f}%")
                print(f"   Val-Train Loss Gap: {overfit_check['loss_gap']:.4f}")
        
        # Epoch summary
        epoch_time = time.time() - epoch_start
        print(f"\nEpoch {epoch+1} Summary:")
        print(f"  Train: Loss={train_loss:.4f}, Acc={train_acc:.2f}%")
        print(f"  Val:   Loss={val_loss:.4f}, Acc={val_acc:.2f}%")
        print(f"  Val:   F1={val_f1:.2f}%, Prec={val_precision:.2f}%, Rec={val_recall:.2f}%")
        print(f"  Composite Score: {composite_score:.2f}")
        if is_best:
            print(f"  🎯 NEW BEST MODEL!")
        print(f"  LR: {optimizer.param_groups[0]['lr']:.6f} | Time: {epoch_time:.1f}s")
        print("=" * 70)

except KeyboardInterrupt:
    print("\n\n⚠️  TRAINING INTERRUPTED")
    print(f"Completed {epoch + 1}/{NUM_EPOCHS} epochs")
    if best_epoch > 0:
        print(f"Best model saved at epoch {best_epoch}")

# Save final checkpoint
final_metrics = {
    'train_loss': train_loss, 'train_acc': train_acc,
    'val_loss': val_loss, 'val_acc': val_acc,
    'val_f1': val_f1, 'composite_score': composite_score
}

save_checkpoint(model, optimizer, epoch + 1, final_metrics, False,
              CHECKPOINT_DIR, SERIAL_NUMBER, MODEL_NAME, SEED, 'last')

# Training complete
total_time = time.time() - start_time
hours = int(total_time // 3600)
minutes = int((total_time % 3600) // 60)

print("\n" + "="*80)
print("TRAINING COMPLETE")
print("="*80)

if best_epoch > 0:
    print(f"Best model: Epoch {best_epoch}")
    print(f"  Composite Score: {best_composite_score:.2f}")
    print(f"  Val Accuracy: {best_val_acc:.2f}%")
    print(f"  Val Loss: {best_val_loss:.4f}")

print(f"\nTotal training time: {hours}h {minutes}m")
print(f"Serial: {SERIAL_NUMBER:02d}")
print(f"Checkpoints: {CHECKPOINT_DIR}")
print("="*80)

# Save training history
history_file = CHECKPOINT_DIR / f"{SERIAL_NUMBER:02d}_{MODEL_NAME}_seed{SEED}_history.json"
with open(history_file, 'w') as f:
    history_serializable = {k: [float(x) if isinstance(x, (np.floating, np.integer)) else x 
                                for x in v] if isinstance(v, list) else v 
                           for k, v in history.items()}
    json.dump(history_serializable, f, indent=2)

print(f"\nTraining history saved: {history_file.name}")
print("\nUse Master_Evaluation.ipynb for test set evaluation")
print("="*80)


STARTING TRAINING - CBAM_RESNET
Serial: 09 | Seed: 126 | Epochs: 50
Device: cuda
Val size: 8,369 images

Epoch [1/50]
----------------------------------------------------------------------


Saved best: 09_cbam_resnet_seed126_epoch1_best_20260115_110530.pth

Epoch 1 Summary:
  Train: Loss=0.4800, Acc=85.83%
  Val:   Loss=0.3895, Acc=92.62%
  Val:   F1=86.90%, Prec=87.84%, Rec=86.40%
  Composite Score: 90.96
  🎯 NEW BEST MODEL!
  LR: 0.001000 | Time: 142.5s

Epoch [2/50]
----------------------------------------------------------------------


Saved best: 09_cbam_resnet_seed126_epoch2_best_20260115_110750.pth

Epoch 2 Summary:
  Train: Loss=0.3481, Acc=89.62%
  Val:   Loss=0.4091, Acc=92.30%
  Val:   F1=86.42%, Prec=86.93%, Rec=86.00%
  Composite Score: 91.90
  🎯 NEW BEST MODEL!
  LR: 0.001000 | Time: 139.1s

Epoch [3/50]
----------------------------------------------------------------------



Epoch 3 Summary:
  Train: Loss=0.3210, Acc=90.33%
  Val:   Loss=0.4235, Acc=84.84%
  Val:   F1=78.13%, Prec=76.24%, Rec=84.97%
  Composite Score: 85.97
  LR: 0.001000 | Time: 142.8s

Epoch [4/50]
----------------------------------------------------------------------



Epoch 4 Summary:
  Train: Loss=0.2907, Acc=91.22%
  Val:   Loss=0.2984, Acc=88.55%
  Val:   F1=83.54%, Prec=81.63%, Rec=90.51%
  Composite Score: 89.91
  LR: 0.001000 | Time: 137.8s

Epoch [5/50]
----------------------------------------------------------------------


...


Epoch 46 Summary:
  Train: Loss=0.1092, Acc=96.30%
  Val:   Loss=0.1357, Acc=95.63%
  Val:   F1=92.58%, Prec=90.69%, Rec=95.52%
  Composite Score: 95.92
  LR: 0.000250 | Time: 137.8s

Epoch [47/50]
----------------------------------------------------------------------



Epoch 47 Summary:
  Train: Loss=0.1075, Acc=96.33%
  Val:   Loss=0.1425, Acc=95.71%
  Val:   F1=92.76%, Prec=91.15%, Rec=94.67%
  Composite Score: 96.00
  LR: 0.000250 | Time: 138.0s

Epoch [48/50]
----------------------------------------------------------------------



Epoch 48 Summary:
  Train: Loss=0.1044, Acc=96.40%
  Val:   Loss=0.1270, Acc=95.91%
  Val:   F1=93.02%, Prec=91.08%, Rec=95.69%
  Composite Score: 96.22
  LR: 0.000250 | Time: 138.3s

Epoch [49/50]
----------------------------------------------------------------------



Epoch 49 Summary:
  Train: Loss=0.1064, Acc=96.36%
  Val:   Loss=0.1252, Acc=95.72%
  Val:   F1=92.80%, Prec=90.73%, Rec=95.82%
  Composite Score: 96.05
  LR: 0.000250 | Time: 138.4s

Epoch [50/50]
----------------------------------------------------------------------


Saved intermediate: 09_cbam_resnet_seed126_epoch50_intermediate_20260115_125846.pth

Epoch 50 Summary:
  Train: Loss=0.1025, Acc=96.46%
  Val:   Loss=0.1353, Acc=96.01%
  Val:   F1=93.16%, Prec=91.55%, Rec=95.27%
  Composite Score: 96.29
  LR: 0.000250 | Time: 138.5s
Saved last: 09_cbam_resnet_seed126_epoch50_last_20260115_125847.pth

TRAINING COMPLETE
Best model: Epoch 45
  Composite Score: 96.56
  Val Accuracy: 96.25%
  Val Loss: 0.1376

Total training time: 1h 55m
Serial: 09
Checkpoints: C:\Users\Ajant\Documents\Project MSc - CSC-40098\3 - Experiments\Notebooks\Model_Training\Checkpoints

Training history saved: 09_cbam_resnet_seed126_history.json

Use Master_Evaluation.ipynb for test set evaluation
